# Implementation of YOLOv12 trained LiDar dataset. focus on reflective properties

Every snowpole has a reflective piece on it, the training takes this in to concideration, focusing on brigth small areas in a dark environment. Placed on in the middel of every pole

In [ ]:
!git clone https://github.com/sunsmarterjie/yolov12
%cd yolov12
%pip install roboflow supervision flash-attn --upgrade -q
%pip install -r requirements.txt
%pip install -e .
%pip install --upgrade flash-attn

# Not in tutorial but necesarry:
%pip install huggingface_hub ultralytics

In [1]:
dataset_path_lidar = "/home/edvarsa/pole_detection/Poles/lidar/"

In [4]:
from ultralytics import YOLO

# Initialize YOLOv12 model
model = YOLO('yolov12s.yaml')

# Define custom search space optimized for small bright reflections in blue areas
custom_space = {
    # Augmentation parameters
    "hsv_h": (0.0, 0.05),   # Even more minimal hue shift to preserve blue tones
    "hsv_s": (0.0, 0.2),    # Very low saturation range to maintain white points
    "hsv_v": (0.0, 0.15),   # Minimal value changes to preserve bright spots
    "degrees": (0.0, 5.0),   # Reduced rotation to keep small features intact
    "scale": (0.3, 0.7),    # Increased minimum scale for better small object focus
    "mosaic": (0.0, 0.4),   # Further reduced mosaic to preserve tiny features
    "mixup": (0.0, 0.15),   # Minimal mixup to maintain pixel-level features
    
    # Loss weights
    "box": (10.0, 20.0),    # Even higher box loss for single-pixel precision
    "cls": (0.2, 0.8),      # Adjusted classification loss for binary detection
    "dfl": (2.0, 4.0),      # Increased DFL range for better localization
    
    
}

# Run hyperparameter tuning with adjusted parameters
results = model.tune(
    data=f'{dataset_path_lidar}labels/data.yaml',
    space=custom_space,
    epochs=250,
    val=True,
    plots=True,
    device='0',
    seed=42,
    use_ray=False,
    patience=15,
    amp=False,
    batch=8,
    workers=4
)

Tuner: Initialized Tuner instance with 'tune_dir=/home/edvarsa/pole_detection/TDT4265-Snow-pole-detection/yolov12_our_files/yolov12/yolov12/yolov12/yolov12/runs/detect/tune3'
Tuner: 💡 Learn about tuning at https://docs.ultralytics.com/guides/hyperparameter-tuning
Tuner: Starting iteration 1/10 with hyperparameters: {'hsv_h': 0.015, 'hsv_s': 0.2, 'hsv_v': 0.15, 'degrees': 0.0, 'scale': 0.5, 'mosaic': 0.4, 'mixup': 0.0, 'box': 10.0, 'cls': 0.5, 'dfl': 2.0}


/home/edvarsa/.local/lib/python3.12/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


New https://pypi.org/project/ultralytics/8.3.113 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.63 🚀 Python-3.12.3 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4090, 24069MiB)
engine/trainer: task=detect, mode=train, model=yolov12s.yaml, data=/home/edvarsa/pole_detection/Poles/lidar/labels/data.yaml, epochs=250, time=None, patience=15, batch=8, imgsz=640, save=True, save_period=-1, cache=False, device=0, workers=4, project=None, name=train13, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=42, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=False, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_

E0000 00:00:1745333595.027550   81460 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745333595.029185   81460 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Overriding model.yaml nc=80 with nc=1

                   from  n    params  module                                       arguments                     
  0                  -1  1       928  ultralytics.nn.modules.conv.Conv             [3, 32, 3, 2]                 
  1                  -1  1      9344  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2, 1, 2]          
  2                  -1  1     26080  ultralytics.nn.modules.block.C3k2            [64, 128, 1, False, 0.25]     
  3                  -1  1     37120  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2, 1, 4]        
  4                  -1  1    103360  ultralytics.nn.modules.block.C3k2            [128, 256, 1, False, 0.25]    
  5                  -1  1    590336  ultralytics.nn.modules.conv.Conv             [256, 256, 3, 2]              
  6                  -1  2    677120  ultralytics.nn.modules.block.A2C2f           [256, 256, 2, True, 4]        
  7                  -1  1   1180672  ultralytics

train: Scanning /home/edvarsa/pole_detection/Poles/lidar/combined_color/train.cache... 0 images, 1367 backgrounds, 0 corrupt: 100%|██████████| 1367/1367 [00:00<?, ?it/s]
val: Scanning /home/edvarsa/pole_detection/Poles/lidar/combined_color/valid.cache... 0 images, 390 backgrounds, 0 corrupt: 100%|██████████| 390/390 [00:00<?, ?it/s]


WARNING ⚠️ No labels found in /home/edvarsa/pole_detection/Poles/lidar/combined_color/train.cache, training may not work correctly. See https://docs.ultralytics.com/datasets for dataset formatting guidance.
WARNING ⚠️ No labels found in /home/edvarsa/pole_detection/Poles/lidar/combined_color/valid.cache, training may not work correctly. See https://docs.ultralytics.com/datasets for dataset formatting guidance.
Plotting labels to /home/edvarsa/pole_detection/TDT4265-Snow-pole-detection/yolov12_our_files/yolov12/yolov12/yolov12/yolov12/runs/detect/train13/labels.jpg... 
zero-size array to reduction operation maximum which has no identity
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 121 weight(decay=0.0), 128 weight(decay=0.0005), 127 bias(decay=0.0)
TensorBoard: WARNING ⚠️ TensorBoard graph visualization failure Tracing fai

      1/250      4.92G          0      43.28          0          0        640:  78%|███████▊  | 134/171 [00:07<00:02, 17.78it/s]
Traceback (most recent call last):
  File "/home/edvarsa/.local/bin/yolo", line 8, in <module>
    sys.exit(entrypoint())
             ^^^^^^^^^^^^
  File "/home/edvarsa/pole_detection/TDT4265-Snow-pole-detection/yolov12_our_files/yolov12/ultralytics/cfg/__init__.py", line 983, in entrypoint
    getattr(model, mode)(**overrides)  # default args from model
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/edvarsa/pole_detection/TDT4265-Snow-pole-detection/yolov12_our_files/yolov12/ultralytics/engine/model.py", line 808, in train
    self.trainer.train()
  File "/home/edvarsa/pole_detection/TDT4265-Snow-pole-detection/yolov12_our_files/yolov12/ultralytics/engine/trainer.py", line 207, in train
    self._do_train(world_size)
  File "/home/edvarsa/pole_detection/TDT4265-Snow-pole-detection/yolov12_our_files/yolov12/ultralytics/engine/trainer.py", line 409, in _

KeyboardInterrupt: 